# **Day 33: Files and Folders with os and pathlib**

Before reading and writing files, we need to know how to work with **paths** (the address of a file) and **folders**.

- **os:** The older module. Still used in many programs.
- **pathlib:** The newer and easier module. We will mostly use this one.

In [1]:
import os
from pathlib import Path

print(os.name)          # 'nt' means Windows
here = Path.cwd()       # Current working folder
print(type(here))

nt
<class 'pathlib.WindowsPath'>


## **Building Paths**
In `pathlib`, we join parts of a path with `/`. It works on both Windows and Linux, so we do not need to worry about `\` or `/`.

In [2]:
p = Path("data") / "sentinel2" / "B04.tif"
print(p)
print("File name :", p.name)
print("Without extension:", p.stem)
print("Extension :", p.suffix)
print("Parent folder:", p.parent)
print("Parts     :", p.parts)

data\sentinel2\B04.tif
File name : B04.tif
Without extension: B04
Extension : .tif
Parent folder: data\sentinel2
Parts     : ('data', 'sentinel2', 'B04.tif')


In [3]:
# The old way with os
print(os.path.join("data", "sentinel2", "B04.tif"))

data\sentinel2\B04.tif


## **Example from Remote Sensing: Organizing a Project Folder**
A typical remote sensing project has many files. It is good practice to keep them in a clear folder structure. Let us create one.

In [4]:
project = Path("rs_project")

folders = [
    project / "raw" / "sentinel2",
    project / "raw" / "landsat8",
    project / "processed" / "ndvi",
    project / "vector",
    project / "maps",
]
for f in folders:
    f.mkdir(parents=True, exist_ok=True)

# Create some sample (empty) files
for band in ["B02", "B03", "B04", "B08"]:
    (project / "raw" / "sentinel2" / f"T45QXF_20240120_{band}.tif").write_bytes(b"")
(project / "vector" / "kolkata_boundary.geojson").write_text("{}")
(project / "README.txt").write_text("Kolkata NDVI project\n")

print("Project created")

Project created


- `mkdir(parents=True, exist_ok=True)`: Makes the folder, and also any missing parent folders. `exist_ok=True` means no error if it already exists.
- `write_text()` and `write_bytes()` make a file and write into it in one step.

## **Listing Files**

In [5]:
# Everything directly inside a folder
for item in project.iterdir():
    print(item.name, "(folder)" if item.is_dir() else "(file)")

raw (folder)


processed (folder)
vector (folder)
maps (folder)
README.txt (file)


In [6]:
# Find all .tif files anywhere inside (rglob searches all sub-folders)
for f in sorted(project.rglob("*.tif")):
    print(f)

rs_project\raw\sentinel2\T45QXF_20240120_B02.tif
rs_project\raw\sentinel2\T45QXF_20240120_B03.tif
rs_project\raw\sentinel2\T45QXF_20240120_B04.tif
rs_project\raw\sentinel2\T45QXF_20240120_B08.tif


In [7]:
# A small function to print the folder tree
def tree(path, indent=""):
    for item in sorted(path.iterdir()):
        print(indent + ("[+] " if item.is_dir() else " -  ") + item.name)
        if item.is_dir():
            tree(item, indent + "    ")

print(project)
tree(project)

rs_project
[+] maps
[+] processed
    [+] ndvi
[+] raw
    [+] landsat8
    [+] sentinel2
         -  T45QXF_20240120_B02.tif
         -  T45QXF_20240120_B03.tif
         -  T45QXF_20240120_B04.tif
         -  T45QXF_20240120_B08.tif
 -  README.txt
[+] vector
     -  kolkata_boundary.geojson


The `tree()` function calls itself for every folder. This is the recursion we learned on Day 27.

## **Checking, Renaming, Copying and Deleting**

In [8]:
import shutil

f = project / "README.txt"
print(f.exists(), f.is_file())

new = f.rename(project / "ABOUT.txt")
print(new.exists(), f.exists())

shutil.copy(new, project / "ABOUT_backup.txt")
print(sorted(p.name for p in project.iterdir()))

True True
True False
['ABOUT.txt', 'ABOUT_backup.txt', 'maps', 'processed', 'raw', 'vector']


In [9]:
# Delete one file, and then the whole project folder
(project / "ABOUT_backup.txt").unlink()
shutil.rmtree(project)
print("Project folder exists:", project.exists())

Project folder exists: False


**Be careful:** `shutil.rmtree()` deletes a folder and **everything inside it**, without asking. There is no Recycle Bin.

## **Practice Problem: Sort Files by Type**

In [10]:
downloads = Path("downloads_mess")
downloads.mkdir(exist_ok=True)
for name in ["tram.jpg", "result.pdf", "notes.txt", "B04.tif", "B08.tif", "boundary.shp"]:
    (downloads / name).write_text("x")

for f in list(downloads.iterdir()):
    if f.is_file():
        folder = downloads / f.suffix[1:]          # 'jpg', 'pdf', 'tif' ...
        folder.mkdir(exist_ok=True)
        f.rename(folder / f.name)

tree(downloads)
shutil.rmtree(downloads)

[+] jpg
     -  tram.jpg
[+] pdf
     -  result.pdf
[+] shp
     -  boundary.shp
[+] tif
     -  B04.tif
     -  B08.tif
[+] txt
     -  notes.txt


In [11]:
# Quick question
# Guess the output of this code cell
# from pathlib import Path
# p = Path("maps/kolkata_ndvi.png")
# print(p.stem, p.suffix)